# Train the baseline model

## Overall Workflow
1. Read packages from unknown_packages.txt(unknown group) and vulnerable_packages.txt(positive group) and combine them into one collection
2. Initialize a FeatureGenerator with system = "pypi"
3. Iterate through package + version:
    + Generate package features for the package + version using FeatureGenerator's function of get_package_metadata(no structural data)
    + Append the package's embeddings into the tensor/dataframe initialized earlier
4. Fit a clustering Model (K-means) on these package embeddings (Tune K use Elbow Method)
5. Evaluate the clustering result (best K)
    + Define `risk(cluster) = #vulnerable_package / size of cluster`
    + Rank clusters by riskscore
    + Get TopK cluster, for example(10%)
    + Calculate the recall and precision of TopK cluster:
        + Recall: How many known vulnerable package is in TopK cluster?
        + Precision: Out of the TopK cluster, how much of them are known vulnerable packages
6. Save the baseline model

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
import joblib

from scripts.FeatureGenerator import FeatureGenerator

VULNERABLE_FILE  = '../data/vulnerable_packages.txt'
UNKNOWN_FILE     = '../data/unknown_packages.txt'
METADATA_CACHE   = '../data/package_metadata_cache.csv'   # shared across models
FEATURES_CACHE   = '../data/baseline_features.csv'        # labeled dataset for this model
MODEL_OUTPUT     = '../models/lib/baseline_kmeans.pkl'
SCALER_OUTPUT    = '../models/lib/baseline_scaler.pkl'

PACKAGE_FEATURES = [
    'num_authors', 'num_maintainers', 'num_roles', 'has_organization',
    'has_license', 'yanked', 'has_sdist', 'has_sig',
    'num_dependencies', 'has_requires_python',
    'num_releases', 'days_since_first_release', 'days_since_last_release', 'dev_status_level',
    'description_length', 'num_classifiers', 'num_project_urls', 'has_homepage',
    'num_wheel_dists', 'total_dist_size_kb',
]

os.makedirs('../models/lib', exist_ok=True)
print('Setup complete.')

Setup complete.


## Step 1-2: Load packages and assign labels

In [2]:
def load_packages(filepath, label):
    entries = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                pkg, ver = line.rsplit('@', 1)
                entries.append({'package': pkg, 'version': ver, 'label': label})
    return entries

vulnerable_entries = load_packages(VULNERABLE_FILE, label=1)
unknown_entries    = load_packages(UNKNOWN_FILE,    label=0)
all_entries        = vulnerable_entries + unknown_entries

print(f'Vulnerable : {len(vulnerable_entries)}')
print(f'Unknown    : {len(unknown_entries)}')
print(f'Total      : {len(all_entries)}')

Vulnerable : 5000
Unknown    : 5000
Total      : 10000


## Step 3-4: Feature extraction

Calls the PyPI API for each package. Results are cached to `baseline_features.csv` — re-run this cell only when you want to refresh the data. Otherwise skip to the **Load cache** cell below.

In [3]:
fg = FeatureGenerator(system='pypi', cache_path=METADATA_CACHE)

rows = []
for entry in tqdm(all_entries, desc='Extracting features'):
    pkg, ver, label = entry['package'], entry['version'], entry['label']
    try:
        metadata = fg.get_package_metadata(pkg, ver)
        if not metadata:
            continue
        row = {f: metadata.get(f, None) for f in PACKAGE_FEATURES}
        row['package'] = pkg
        row['version'] = ver
        row['label']   = label
        rows.append(row)
    except Exception:
        pass

features_df = pd.DataFrame(rows)
features_df.to_csv(FEATURES_CACHE, index=False)
print(f'Extracted {len(features_df)} rows  ({features_df["label"].sum():.0f} vulnerable).')
features_df.head()

Extracting features: 100%|██████████| 10000/10000 [1:00:59<00:00,  2.73it/s]

Extracted 9937 rows  (5000 vulnerable).


,num_authors,num_maintainers,num_roles,has_organization,has_license,yanked,has_sdist,has_sig,num_dependencies,has_requires_python,...,dev_status_level,description_length,num_classifiers,num_project_urls,has_homepage,num_wheel_dists,total_dist_size_kb,package,version,label
0,1,0,8,0,1,0,1,0,12,0,...,0,166,0,1,1,1,257,distributed,1.10.2,1
1,1,0,2,0,1,0,1,0,24,1,...,3,3770,16,7,1,1,17630,streamlit,1.14.1,1
2,1,0,3,0,1,0,1,0,22,1,...,2,1714,7,6,1,1,26443,homeassistant,2022.4.7,1
3,1,0,1,0,1,0,1,0,38,1,...,2,11541,17,5,1,1,4927,py7zr,0.20.0,1
4,1,0,1,0,1,0,1,0,4,0,...,0,10358,1,1,1,1,169,openzeppelin-cairo-contracts,0.5.0,1


In [4]:
# Load cache (run this instead of the extraction cell if features already exist)
features_df = pd.read_csv(FEATURES_CACHE)
print(f'Loaded {len(features_df)} rows  ({features_df["label"].sum():.0f} vulnerable).')
features_df.describe()

Loaded 9937 rows  (5000 vulnerable).


,num_authors,num_maintainers,num_roles,has_organization,has_license,yanked,has_sdist,has_sig,num_dependencies,has_requires_python,...,days_since_first_release,days_since_last_release,dev_status_level,description_length,num_classifiers,num_project_urls,has_homepage,num_wheel_dists,total_dist_size_kb,label
count,9937.000000,9937.000000,9937.000000,9937.000000,9937.000000,9937.000000,9937.000000,9937.0,9937.000000,9937.000000,...,9937.000000,9937.000000,9937.000000,9937.000000,9937.000000,9937.000000,9937.000000,9937.000000,9.937000e+03,9937.000000
mean,0.880749,0.081916,1.856194,0.112207,0.732012,0.007950,0.928650,0.0,21.873604,0.642951,...,2339.513938,693.862333,1.076884,5557.009158,7.276743,1.903693,0.854081,1.385831,1.574035e+04,0.503170
std,0.706942,0.317137,2.006619,0.315637,0.442934,0.088813,0.257421,0.0,86.190872,0.479154,...,1556.327017,1055.737161,1.259156,11820.674764,5.996249,1.858274,0.353043,3.863444,1.503051e+05,0.500015
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
25%,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.0,1.000000,0.000000,...,1070.000000,9.000000,0.000000,315.000000,3.000000,1.000000,1.000000,1.000000,1.700000e+01,0.000000
50%,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.0,6.000000,1.000000,...,2271.000000,155.000000,0.000000,2361.000000,7.000000,1.000000,1.000000,1.000000,1.340000e+02,1.000000
75%,1.000000,0.000000,2.000000,0.000000,1.000000,0.000000,1.000000,0.0,20.000000,1.000000,...,3271.000000,1037.000000,2.000000,5960.000000,11.000000,2.000000,1.000000,1.000000,2.820000e+03,1.000000
max,15.000000,7.000000,24.000000,1.000000,1.000000,1.000000,1.000000,0.0,1541.000000,1.000000,...,7484.000000,7475.000000,3.000000,309870.000000,39.000000,10.000000,1.000000,114.000000,5.829051e+06,1.000000


## Preprocessing

- Replace sentinel value `-1` (no release history available) with column median.
- Impute any remaining NaN with median.
- StandardScale all features before clustering.

In [5]:
X_raw = features_df[PACKAGE_FEATURES].copy()
labels = features_df['label'].values

# -1 is a sentinel for "no release history data" — treat as missing
X_raw.replace(-1, np.nan, inplace=True)

# Impute missing values with column median
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)

# Standardize for K-Means (sensitive to scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f'Feature matrix shape : {X_scaled.shape}')
print(f'NaN remaining        : {np.isnan(X_scaled).sum()}')

missing_pct = X_raw.isna().mean().sort_values(ascending=False)
print('\nMissing % per feature (before imputation):')
print(missing_pct[missing_pct > 0].to_string())

Feature matrix shape : (9937, 20)
NaN remaining        : 0

Missing % per feature (before imputation):
Series([], )


## Step 5: Train K-Means — Elbow Method

Sweep K from 2 to 20. The elbow is detected as the K where the **marginal gain** in inertia reduction drops most sharply (largest second-difference).

In [6]:
K_RANGE = range(2, 21)

inertias = {}
for k in tqdm(K_RANGE, desc='Elbow sweep'):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias[k] = km.inertia_

elbow_df = pd.DataFrame({
    'K'       : list(inertias.keys()),
    'Inertia' : list(inertias.values()),
})
elbow_df['Delta']       = elbow_df['Inertia'].diff().abs()        # 1st difference
elbow_df['Delta2']      = elbow_df['Delta'].diff().abs()          # 2nd difference (elbow signal)
elbow_df['Gain_pct']    = (elbow_df['Delta'] / elbow_df['Inertia'].shift(1) * 100).round(2)

print(elbow_df.to_string(index=False))

# Auto-detect elbow: K where second-difference is maximised
best_k = int(elbow_df.loc[elbow_df['Delta2'].idxmax(), 'K'])
print(f'\nAuto-detected elbow K = {best_k}')

Elbow sweep: 100%|██████████| 19/19 [00:01<00:00, 10.72it/s]

 K       Inertia        Delta      Delta2  Gain_pct
 2 164840.969577          NaN         NaN       NaN
 3 154835.359310 10005.610266         NaN      6.07
 4 145582.727043  9252.632267  752.978000      5.98
 5 136243.278590  9339.448453   86.816186      6.42
 6 129487.451985  6755.826605 2583.621848      4.96
 7 120201.491520  9285.960466 2530.133861      7.17
 8 114265.897448  5935.594072 3350.366394      4.94
 9 105234.003536  9031.893912 3096.299840      7.90
10  98363.392738  6870.610798 2161.283114      6.53
11  94350.421184  4012.971554 2857.639243      4.08
12  90070.289065  4280.132119  267.160565      4.54
13  82469.881617  7600.407448 3320.275328      8.44
14  77004.163236  5465.718381 2134.689067      6.63
15  72695.155940  4309.007296 1156.711085      5.60
16  71302.754602  1392.401338 2916.605958      1.92
17  67017.073197  4285.681405 2893.280067      6.01
18  65126.255486  1890.817711 2394.863694      2.82
19  63615.997790  1510.257696  380.560015      2.32
20  62363.03

In [7]:
# Override best_k here if you disagree with the auto-detection
# best_k = 10

final_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_km.fit_predict(X_scaled)

features_df['cluster'] = cluster_labels
print(f'Trained KMeans with K={best_k}')
print(features_df.groupby('cluster')['label'].agg(['sum', 'count']).rename(
    columns={'sum': 'n_vulnerable', 'count': 'cluster_size'}
))

Trained KMeans with K=8
         n_vulnerable  cluster_size
cluster                            
0                1741          1791
1                 872          2353
2                 609          1864
3                 693           694
4                 790          1676
5                  84           103
6                 166          1377
7                  45            79


## Step 6: Evaluate

**Risk score** per cluster = `n_vulnerable / cluster_size`  
**Top-K clusters** = top 10% of clusters ranked by risk score (configurable via `TOP_K_PCT`)

Metrics:
- **Recall** — what fraction of all known vulnerable packages fall in the top-K clusters?
- **Precision** — of everything inside the top-K clusters, what fraction is actually vulnerable?

In [8]:
TOP_K_PCT = 0.10   # top 10% of clusters considered "high risk"

# --- Compute risk score per cluster ---
cluster_stats = (
    features_df.groupby('cluster')['label']
    .agg(n_vulnerable='sum', cluster_size='count')
    .assign(risk_score=lambda df: df['n_vulnerable'] / df['cluster_size'])
    .sort_values('risk_score', ascending=False)
    .reset_index()
)

print('=== All clusters ranked by risk score ===')
print(cluster_stats.to_string(index=False))

# --- Select top-K clusters ---
n_top = max(1, round(best_k * TOP_K_PCT))
top_clusters = set(cluster_stats.head(n_top)['cluster'])
print(f'\nTop-{n_top} cluster(s) (top {TOP_K_PCT*100:.0f}%): {top_clusters}')

# --- Recall & Precision ---
total_vulnerable  = (features_df['label'] == 1).sum()
in_top_mask       = features_df['cluster'].isin(top_clusters)

true_positives    = ((features_df['label'] == 1) & in_top_mask).sum()
top_cluster_size  = in_top_mask.sum()

recall    = true_positives / total_vulnerable if total_vulnerable else 0
precision = true_positives / top_cluster_size if top_cluster_size else 0
f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

print(f'\n--- Evaluation (top {TOP_K_PCT*100:.0f}% clusters as "risky") ---')
print(f'Total packages          : {len(features_df)}')
print(f'Total vulnerable        : {total_vulnerable}')
print(f'Top cluster size        : {top_cluster_size}')
print(f'True positives          : {true_positives}')
print(f'Recall                  : {recall:.4f}  ({recall*100:.1f}%)')
print(f'Precision               : {precision:.4f}  ({precision*100:.1f}%)')
print(f'F1                      : {f1:.4f}')

=== All clusters ranked by risk score ===
 cluster  n_vulnerable  cluster_size  risk_score
       3           693           694    0.998559
       0          1741          1791    0.972083
       5            84           103    0.815534
       7            45            79    0.569620
       4           790          1676    0.471360
       1           872          2353    0.370591
       2           609          1864    0.326717
       6           166          1377    0.120552

Top-1 cluster(s) (top 10%): {3}

--- Evaluation (top 10% clusters as "risky") ---
Total packages          : 9937
Total vulnerable        : 5000
Top cluster size        : 694
True positives          : 693
Recall                  : 0.1386  (13.9%)
Precision               : 0.9986  (99.9%)
F1                      : 0.2434


## Step 7: Save the baseline model

In [9]:
joblib.dump(final_km, MODEL_OUTPUT)
joblib.dump(scaler,   SCALER_OUTPUT)

# Save cluster risk-score table for downstream use
cluster_stats.to_csv('../models/lib/cluster_risk_scores.csv', index=False)

print(f'Model  saved → {MODEL_OUTPUT}')
print(f'Scaler saved → {SCALER_OUTPUT}')
print(f'Cluster risk scores saved → ../models/lib/cluster_risk_scores.csv')

Model  saved → ../models/lib/baseline_kmeans.pkl
Scaler saved → ../models/lib/baseline_scaler.pkl
Cluster risk scores saved → ../models/lib/cluster_risk_scores.csv
